In [ ]:
# tests/test_mask_correctness.py
import numpy as np
import pandas as pd
from helpers.preprocessor import Preprocessor
from imputers.generate_mask import generate_mask
from helpers.mask_mapping import mask_original_to_encoded

def check_mask(df: pd.DataFrame, prepper: Preprocessor):
    # 1) Shapes
    masks = generate_mask(df, mask_type="MCAR", mask_num=3, p=0.3, exclude_cols=[], random_state=123)
    assert len(masks) == 3
    for m in masks:
        assert m.shape == (len(df), len(df.columns))

    # 2) Mapping shapes
    m0_enc = mask_original_to_encoded(prepper, masks[0])
    Denc = len(prepper.num_idx) + len(prepper.cat_idx)
    assert m0_enc.shape == (len(df), Denc)

    # 3) Encode/Decode idempotence on “no-missing” path
    X_ord = prepper.encodeDf("Ordinal", df)
    # decode then re-encode — just checks shape/ordering consistency, not exact roundtrip of categories
    dec = prepper.decodeNp("Ordinal", X_ord)
    assert dec.shape == X_ord.shape  # encoded width preserved during decode/encode flow

    # 4) Semantics True==MISSING
    # mask 30% cells; ensure empirical rate ~p on maskable subset
    m_all = masks[0]
    maskable = np.ones_like(m_all, dtype=bool)
    # if you exclude some columns: set those positions False in `maskable`
    emp = m_all[maskable].mean()
    assert 0.15 <= emp <= 0.45, f"Empirical missing {emp:.3f} far from target"

    print("Mask generation + mapping checks passed.")

if __name__ == "__main__":
    # tiny demo df
    df = pd.DataFrame({
        "num1": [1,2,3,4,5],
        "num2": [10,20,30,40,50],
        "cat":  ["a","b","a","b","a"],
        "path": ["/a","/b","/c","/d","/e"],
    })
    # Suppose your Preprocessor excludes 'path' automatically
    prepper = Preprocessor(dataname="Scenario33", data_dir="/Users/yuandouwang/Documents/projects/6G-Data-process/6GDALI_Datasets/DeepSense/Scenario33/")
    check_mask(df, prepper)


2025-11-12T11:06:13.981399Z [error    ] Unable to retrieve variable from secrets backend (MetastoreBackend). Checking subsequent secrets backend. [airflow.models.variable] loc=variable.py:449
Traceback (most recent call last):
  File "/Users/yuandouwang/miniconda3/envs/dataImputation/lib/python3.12/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
  File "/Users/yuandouwang/miniconda3/envs/dataImputation/lib/python3.12/site-packages/sqlalchemy/engine/default.py", line 951, in do_execute
    cursor.execute(statement, parameters)
sqlite3.OperationalError: no such table: variable

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/yuandouwang/miniconda3/envs/dataImputation/lib/python3.12/site-packages/airflow/models/variable.py", line 445, in get_variable_from_secrets
    var_val = secrets_backend.get_variable(key=key)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyError: 'Variable N2N_S3_ENDPOINT does not exist'